In [41]:
from ccdc import io
from ccdc.io import EntryReader

from collections import defaultdict
from itertools import islice

from tqdm import tqdm

import pandas as pd

import time

from concurrent.futures import ProcessPoolExecutor

In [42]:
csd = EntryReader('CSD')
csd_reader = io.EntryReader('CSD')

### 1. Extract all records to single CSV table

In [43]:
all_df = []

start = time.time()

total_entry = 500
for entry in tqdm(islice(csd, total_entry), total=total_entry, desc="Traitement"):

    # check organic
    is_organic = entry.is_organic

    # check num components
    num_component = len(entry.molecule.components)

    # check metal
    has_metal = any(atom.is_metal for atom in entry.molecule.atoms)

    # add result
    all_df.append({
        "ID":entry.identifier,
        "SMILES": entry.molecule.smiles,
        "IS_ORGANIC": is_organic,
        "HAS_METAL":has_metal,
        "NUM_COMPONENT": num_component,
    })

delta_time = round(time.time() - start, 1)

all_df = pd.DataFrame(all_df)
all_df.to_csv("csd_all.csv", index=False)

print(f"Sequential : Processed {len(all_df)//1000}k entries in {delta_time}s")

Traitement: 100%|████████████████████████████| 500/500 [00:02<00:00, 168.93it/s]

Sequential : Processed 0k entries in 3.0s


In [44]:
all_df

,ID,SMILES,IS_ORGANIC,HAS_METAL,NUM_COMPONENT
0,AABHTZ,CC(=O)NN1C=NN=C1N(N=Cc1c(Cl)cccc1Cl)C(C)=O,True,False,1
1,AACANI10,[OH2][Ni]123OC(=O)CN41CCCN2(CCC4)CC(=O)O3.O.O,False,True,3
2,AACANI11,[OH2][Ni]123OC(=O)CN41CCCN2(CCC4)CC(=O)O3.O.O,False,True,3
3,AACFAZ,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
4,AACFAZ10,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
...,...,...,...,...,...
495,ABAZPT,CC(=O)N1=NC(=[NH][Pt]21[NH]=C(N=N2C(C)=O)C(C)(...,False,True,1
496,ABAZUA,Cc1cc(C)c(c(C)c1)B1(c2ccccc2c2ccc(cn12)c1ccc(c...,True,False,1
497,ABAZUB,O.COc1cc(Nc2c(cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc23...,True,False,2
498,ABAZUB01,COc1cc(Nc2c(cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc23)C...,True,False,2


In [55]:
start = time.time()

def chunk_extraction(chunk_range):
    start, stop = chunk_range
    results = []
    with io.EntryReader('CSD') as reader:
        for i in range(start, stop):
            entry = reader[i]
            mol = entry.molecule
            results.append({
                "ID": entry.identifier,
                "SMILES": mol.smiles,
                "IS_ORGANIC": entry.is_organic,
                "HAS_METAL": any(atom.is_metal for atom in mol.atoms),
                "NUM_COMPONENT": len(mol.components),
            })
    return results

with io.EntryReader('CSD') as reader:
    NUMBER_ENTRIES = len(reader)

# NUMBER_ENTRIES = 50000
NUM_WORKERS = 24
CHUNK_SIZE = 500
chunk_ranges = [(i, min(i + CHUNK_SIZE, NUMBER_ENTRIES)) for i in range(0, NUMBER_ENTRIES, CHUNK_SIZE)]
pbar = tqdm(total=NUMBER_ENTRIES, unit="entry", desc=f"Extraction ({NUM_WORKERS} workers)", smoothing=0.1)

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
    all_results = []
    for chunk_results in executor.map(chunk_extraction, chunk_ranges):
        all_results.extend(chunk_results)
        pbar.update(len(chunk_results))
pbar.close()

delta_time = round(time.time() - start, 1)

all_df = pd.DataFrame(all_results)
all_df.to_csv("csd_all.csv", index=False)

print(f"Processed {len(all_df)//1000}k entries in {delta_time}s ({NUM_WORKERS} workers)")

Extraction (24 workers): 100%|███| 1436119/1436119 [09:32<00:00, 2508.23entry/s]


Processed 1436k entries in 572.6s (24 workers)


In [28]:
all_df

,ID,SMILES,IS_ORGANIC,HAS_METAL,NUM_COMPONENT
0,AABHTZ,CC(=O)NN1C=NN=C1N(N=Cc1c(Cl)cccc1Cl)C(C)=O,True,False,1
1,AACANI10,[OH2][Ni]123OC(=O)CN41CCCN2(CCC4)CC(=O)O3.O.O,False,True,3
2,AACANI11,[OH2][Ni]123OC(=O)CN41CCCN2(CCC4)CC(=O)O3.O.O,False,True,3
3,AACFAZ,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
4,AACFAZ10,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
...,...,...,...,...,...
499995,IYUYUY,CCCCCCC1c2cc3C(CCCCCC)c4cc5C(CCCCCC)c6cc7C(CCC...,True,False,6
499996,IYUYUZ,CC1=CNC(=O)NC1=O.O,True,False,2
499997,IYUYUZ01,CC1=CNC(=O)NC1=O.O,True,False,2
499998,IYUYUZ02,CC1=CNC(=O)NC1=O.O,True,False,2


### 2. Select only organic one component molecules

In [42]:
all_df = pd.read_csv("csd_all.csv")

df_filtered = all_df[
    (all_df["IS_ORGANIC"] == True) &
    (all_df["HAS_METAL"] == False) &
    (all_df["NUM_COMPONENT"] == 1)
]

print(f"Selected {len(df_filtered)} entries")

Selected 1423 entries


In [43]:
df_filtered

,ID,SMILES,IS_ORGANIC,HAS_METAL,NUM_COMPONENT
0,AABHTZ,CC(=O)NN1C=NN=C1N(N=Cc1c(Cl)cccc1Cl)C(C)=O,True,False,1
3,AACFAZ,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
4,AACFAZ10,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
6,AACMHX10,CC(=O)OC(=C1CCCCC1c1ccccc1)c1ccccc1,True,False,1
19,AAMAND,COC1CCC2(C)C(CCC3C2CCC2(C)C3CC2C(C)=O)C1,True,False,1
...,...,...,...,...,...
4978,ACIZES,CC(C)c1cc(C(C)C)c(c(c1)C(C)C)[Si](=P)[Si](C(C)...,True,False,1
4979,ACIZET,FC(F)(F)c1cccc(NC(=S)NC2CCCCC2)c1,True,False,1
4989,ACIZOC,CC(C)[Si]1(O[Si](P[Si](O[Si](P1)(C(C)C)C(C)C)(...,True,False,1
4990,ACIZOD,CC1=NN(c2ccccc2)C2=C1C1(C)C(COc3ccccc13)CO2,True,False,1


### 3. Count the number of crystal forms

In [44]:
groups = defaultdict(list)
for entry_id in tqdm(df_filtered["ID"]):

    mol = csd_reader.molecule(entry_id)
    key = mol.generate_inchi().inchi

    if key:  # skip if missing
        groups[key].append(mol.identifier)

print(f"Selected {len(groups)} molecules with InChIKey")

100%|██████████████████████████████████████████████████████████████████████████████| 1423/1423 [00:03<00:00, 456.10it/s]

Selected 1192 molecules with InChIKey


In [45]:
df_filtered = df_filtered.set_index("ID")

df_counted = []

for key, entries in groups.items():
    num_forms = len(entries)

    df_counted.append({
        "SMILES": df_filtered.loc[entries[0]]["SMILES"],
        "NUM_FORMS": len(entries)
    })

df_counted = pd.DataFrame(df_counted)
df_counted = df_counted.drop_duplicates("SMILES")

print(f"Selected {len(df_counted)} molecules with counted forms")

Selected 1175 molecules with counted forms


In [46]:
n_total = len(df_counted)
n_mono = sum(df_counted["NUM_FORMS"] == 1)
n_poly = sum(df_counted["NUM_FORMS"] > 1)

print(f"Total molecules: {n_total}")
print(f"Monomorphs: {n_mono} ({round(100 * n_mono / n_total, 1)} %)")
print(f"Polymorphs: {n_poly} ({round(100 * n_poly / n_total, 1)} %)")

Total molecules: 1175
Monomorphs: 1104 (94.0 %)
Polymorphs: 71 (6.0 %)


In [48]:
df_counted.to_csv("csd_counted.csv", index=False)

### 4. Properties

In [72]:
entry = csd[0]
properties = dir(entry)
entry.ccdc_number?
properties

['CrossReference',
 '_CENTIGRADE',
 '_CifAttributes',
 '_KELVIN',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_diagram_is_polymeric',
 '_entry',
 '_experimental_info',
 '_fetch_datum',
 '_hydrogen_treatment_map',
 '_hydrogen_treatment_map_inverse',
 '_make_citation',
 '_make_crystal',
 '_make_diagram',
 '_melting_point_parser',
 '_radiation_source_map',
 '_source_file_availability_map',
 '_source_file_availability_map_inverse',
 'analogue',
 'attributes',
 'bioactivity',
 'calculated_density',
 'calculated_properties',
 'ccdc_number',
 'chemical_name',
 'chemical_name_as_html',
 'color',
 'component_inchis',
 'cross_references',
 'crystal',
 'data

Type:        property
String form: <property object at 0x7ea3f539ade0>
Docstring:  
The CCDC deposition number.

>>> from ccdc.io import EntryReader
>>> entry_reader = EntryReader('CSD')
>>> abebuf = entry_reader.entry('ABEBUF')
>>> print(abebuf.ccdc_number)
241370

In [101]:
BANNED_LIST = ['melting_point_display_string', 'input_melting_point_range', 'formatted_melting_point_text', 'formatted_melting_point_range' ]
MAX_ENTRIES_TEST = 100

prop_list = {}
for prop in dir(entry):
    if prop.startswith('_') or prop in BANNED_LIST:
        continue

    try:
        attr_class = getattr(type(entry), prop, None)
        if isinstance(attr_class, property):
            doc = getattr(attr_class.fget, '__doc__', None)
        else:
            doc = getattr(getattr(entry, prop), '__doc__', None)

        doc_clean = doc.split('>>>')[0].split(':')[0].strip() if doc else None

        example_value = None
        for i, test_entry in enumerate(islice(csd, MAX_ENTRIES_TEST)):
            try:
                value = getattr(test_entry, prop)
                if value:
                    example_value = value
                    break
            except:
                continue

        # if str(example_value)[0] == '<':
        #     continue

        prop_list[prop] = {
            "description": doc_clean,
            "example": str(example_value) if example_value is not None else "No example found"
        }

    except Exception as e:
        if "Solubility Platform" not in str(e):
            print(f"Error with {prop} : {e}")

for prop, infos in prop_list.items():
    if infos['example'][0] == "<":
        print(f"--- {prop} ---")
        print(f"Description : {infos['description']}")
        print(f"Exemple : {infos['example']}")
        print()

print(len(prop_list))

--- CrossReference ---
Description : A cross-reference between entries in the database.
Exemple : <class 'ccdc.entry.Entry.CrossReference'>

--- calculated_properties ---
Description : Returns the calculated properties for the entry, or None if they have not been provided.
Exemple : <ccdc.entry.CrystalCalculatedProperties object at 0x7ea37a522650>

--- crystal ---
Description : The
Exemple : <ccdc.crystal.Crystal object at 0x7ea37a5ede50>

--- disordered_molecule ---
Description : The
Exemple : <ccdc.molecule.Molecule object at 0x7ea37a599d10>

--- from_molecule ---
Description : Construct an entry from a molecule, using the keyword arguments as attributes.
Exemple : <function Entry.from_molecule at 0x7ea3f53a7380>

--- from_string ---
Description : Create an entry from a string representation.

        The format will be auto-detected if not specified.
Exemple : <function Entry.from_string at 0x7ea3f53b00e0>

--- molecule ---
Description : The
Exemple : <ccdc.molecule.Molecule object 

In [108]:
BANNED_LIST = ['melting_point_display_string', 'input_melting_point_range',
               'formatted_melting_point_text', 'formatted_melting_point_range']
MAX_ENTRIES_TEST = 100

entry = csd[0]
molecule = entry.molecule

prop_list = {}
for prop in dir(molecule):
    if prop.startswith('_') or prop in BANNED_LIST:
        continue

    try:
        attr_class = getattr(type(molecule), prop, None)
        if isinstance(attr_class, property):
            doc = getattr(attr_class.fget, '__doc__', None)
        else:
            doc = getattr(getattr(molecule, prop), '__doc__', None)

        doc_clean = doc.split('>>>')[0].split(':')[0].strip() if doc else None

        example_value = None
        for test_entry in islice(csd, MAX_ENTRIES_TEST):
            try:
                value = getattr(test_entry.molecule, prop)
                if value:
                    example_value = value
                    break
            except:
                continue

        # if str(example_value)[0] == '<':
        #     continue
        
        prop_list[prop] = {
            "description": doc_clean,
            "example": str(example_value) if example_value is not None else "No example found"
        }

    except Exception as e:
        if "Solubility Platform" not in str(e):
            print(f"Error with {prop} : {e}")

for prop, infos in prop_list.items():
    print(f"--- {prop} ---")
    print(f"Description : {infos['description']}")
    print(f"Exemple : {infos['example']}")
    print()

print(len(prop_list))

--- Contact ---
Description : A contact between two molecules.
Exemple : <class 'ccdc.molecule.Molecule.Contact'>

--- HBond ---
Description : A hydrogen bond between atoms of a molecule.
Exemple : <class 'ccdc.molecule.Molecule.HBond'>

--- HBondCriterion ---
Description : Defines the conditions for an HBond to be detected in a molecule.

        Contains two dictionary like objects for the donor atom types and the acceptor atom types,
        whether or not hydrogens are required, a distance range and an angle tolerance
        (if hydrogen atoms are required), whether or not the distance range is relative to Van der Waals radii,
        whether inter-, intra- or both HBonds should be detected, and a range of path separation distances for atoms
        to be considered hydrogen bonded.

        In almost all cases the defaults will give good results, but users may wish to fine-tune the detection algorithm.
Exemple : <class 'ccdc.molecule.Molecule.HBondCriterion'>

--- Transformation 